In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# ---------------------------------------------------------
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)

y = (df['Churn'] == 'Yes').astype(int)

num_features = ['tenure', 'MonthlyCharges']
cat_features = [
    'gender', 'Contract', 'PaymentMethod', 'PaperlessBilling',
    'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

X_B = df[num_features + cat_features]

preprocessor_B = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_features)
    ]
)

pipeline_B = Pipeline([
    ('preprocessor', preprocessor_B),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

In [7]:
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
from sklearn.metrics import roc_auc_score
df['y'] = (df['Churn'] == 'Yes').astype(int)
y_np = df['y'].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = cross_val_predict(pipeline_B, X_B, df['y'], cv=cv, method='predict_proba')[:, 1]

In [8]:
df['tenure_bin'] = pd.cut(df['tenure'], bins=range(0, 81, 10), include_lowest=True)
ct = pd.crosstab(df['tenure_bin'], df['Churn'])
chi2, p, dof, _ = chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"chi2={chi2:.1f}, p={p:.2e}, Cramér's V={cramers_v:.3f}")

chi2=920.0, p=2.26e-194, Cramér's V=0.361


In [9]:
rng = np.random.default_rng(42)
aucs = []
for _ in range(500):
    idx = rng.integers(0, len(y_np), len(y_np))
    if len(np.unique(y_np[idx])) < 2:
        continue
    aucs.append(roc_auc_score(y_np[idx], oof[idx]))

lo, hi = np.percentile(aucs, [2.5, 97.5])
print(f"AUC = {roc_auc_score(y_np, oof):.4f}, 95% CI: [{lo:.4f}, {hi:.4f}]")

AUC = 0.8418, 95% CI: [0.8319, 0.8520]


In [10]:
aucs_seed = []
for s in range(5):
    cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=s)
    p_s = cross_val_predict(pipeline_B, X_B, df['y'], cv=cv_s, method='predict_proba')[:, 1]
    aucs_seed.append(roc_auc_score(y_np, p_s))
print(f"AUC: {np.mean(aucs_seed):.4f} ± {np.std(aucs_seed):.4f}")

AUC: 0.8416 ± 0.0003


In [12]:
m = df[df['Contract'] == 'Month-to-month']
t = df[df['Contract'] == 'Two year']
z, p_z = proportions_ztest([m['y'].sum(), t['y'].sum()], [len(m), len(t)])
print(f"Aylıq churn: {m['y'].mean():.3f} | İki illik: {t['y'].mean():.3f} | z={z:.2f}, p={p_z:.2e}")

Aylıq churn: 0.427 | İki illik: 0.028 | z=29.72, p=3.93e-194
